# Interpretability: feature importance & SHAP — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(z): return 1/(1+np.exp(-z))
def ece(confs, correct, bins):
    confs=np.asarray(confs); correct=np.asarray(correct); total=len(confs); out=0.0
    for lo,hi in bins:
        m=(confs>=lo)&(confs<hi if hi<1 else confs<=hi)
        if m.any(): out += m.mean()*abs(correct[m].mean()-confs[m].mean())
    return out
print('setup ok')

## SHAP values decompose one prediction into a baseline plus feature contributions.
This notebook uses one tiny CPU-only toy calculation to visualize Interpretability: feature importance & SHAP. The five cells match the lesson arithmetic: the base components, normalized share, knob sweep, component removal, and final guarded decision.

In [ ]:
# 1) Base calculation for Interpretability: feature importance & SHAP: signed total and absolute mass
vals=np.array([1.2,-0.4,0.9], dtype=float); labels=['a','b','c']
total=float(vals.sum()); abs_total=float(np.abs(vals).sum())
plt.figure(figsize=(6,3)); plt.bar(labels, vals); plt.axhline(0,color='k',lw=.8); plt.title('contribution: total=1.700, abs mass=2.500')
assert abs(total-(1.7))<1e-12 and abs(abs_total-(2.5))<1e-12

In [ ]:
# 2) Normalized leading share: concentration of the explanation or risk
vals=np.array([1.2,-0.4,0.9], dtype=float); shares=np.abs(vals)/np.abs(vals).sum()
plt.figure(figsize=(6,3)); plt.bar(['a','b','c'], shares); plt.ylim(0,1); plt.title('leading share = 0.480')
assert abs(float(shares[0])-(0.48))<1e-12

In [ ]:
# 3) Turn the lesson knob and watch the guarded score move
vals=np.array([1.2,-0.4,0.9], dtype=float); ks=np.linspace(0,0.600,40); guarded=vals.sum()+ks*np.abs(vals).sum()
plt.figure(figsize=(6,3)); plt.plot(ks,guarded); plt.scatter([0.3],[2.45],c='r'); plt.xlabel('knob'); plt.ylabel('guarded score'); plt.title('guarded score at knob=0.300: 2.450')
assert abs(float(vals.sum()+0.3*np.abs(vals).sum())-(2.45))<1e-12

In [ ]:
# 4) Remove the third component and compare with the full result
vals=np.array([1.2,-0.4,0.9], dtype=float); full=float(vals.sum()); contrast=float(vals[:2].sum())
plt.figure(figsize=(6,3)); plt.bar(['full','without c'], [full,contrast], color=['tab:blue','tab:orange']); plt.title('change = -0.900')
assert abs(contrast-(0.7999999999999999))<1e-12 and abs((contrast-full)-(-0.9))<1e-12

In [ ]:
# 5) End-to-end toy decision: raw score, guarded score, and pass line
vals=np.array([1.2,-0.4,0.9], dtype=float); raw=float(vals.sum()); guarded=float(raw+0.3*np.abs(vals).sum()); line=(raw+guarded)/2
plt.figure(figsize=(6,3)); plt.plot([0,1,2],[raw,line,guarded],'-o'); plt.axhline(line,ls='--',color='gray'); plt.xticks([0,1,2],['raw','gate','guarded']); plt.title('the guard changes the deployable score')
assert guarded>raw if 2.5>0 else guarded==raw